## **0. SETUP: IMPORT LIBRARIES AND FILES**

In [1]:
!pip install bertopic
!pip install spacy
!pip install hdbscan
!pip install sentence_transformers

In [2]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
import spacy
import spacy.cli
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from bertopic.vectorizers import ClassTfidfTransformer
from multiprocessing import cpu_count


# Ensure the language model is downloaded
#spacy.cli.download("en_core_web_sm")
# Load spaCy English model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "textcat"])
nlp.max_length = 3000000
# Load all_files database.
df = pd.read_csv("all_files.csv")
texts = df["text"]
doc_list = list(texts)
len(doc_list)

7717

## **1. CLEAN AND PREPROCESS**

#### a. Remove all template text

In [3]:
templatePhrases = []
templatePhrases.append("Include the chapter of the model curriculum, the page number, and line number(s) to ensure that the California Department of Education and Instructional Quality Commission can reference the content of the document when reviewing your comments. Please email this document as a Word document to ethnicstudies@cde.ca.gov. You may contact Kenneth McDonald, Education Programs Consultant, at kmcdonal@cde.ca.gov with any questions regarding this template or the public input process.")
templatePhrases.append("Your Name and Affiliation")
templatePhrases.append("Comment (include page and line numbers where applicable)")
templatePhrases.append("(Download and use to provide specific recommendations)")
templatePhrases.append("2020 Ethnic Studies Model Curriculum May 2019 Draft")
templatePhrases.append("Public Input Template�")
templatePhrases.append('"General" if your comment isn\'t about one')
templatePhrases.append("Curriculum [Enter the Chapter Number here,")
templatePhrases.append('or just "General" for a comment that applies to the')
templatePhrases.append("entire document.]")
templatePhrases.append("[Enter the agency, organization, or business that you represent, if applicable.]")
templatePhrases.append("[Include the page and line number(s) here�Write your comment here]")

def removeTemplatePhrases(text, phraseList):
    result = text
    for phrase in phraseList:
        result = result.replace(phrase, '')
    return result

cleanedDocs = [removeTemplatePhrases(doc, templatePhrases) for doc in doc_list]

#### b. Lemmatize docs
##### *This cell takes 15-20 minutes to run*

In [4]:
lemmatized_docs = []
for spacy_doc in nlp.pipe(cleanedDocs, batch_size=50, n_process=cpu_count()):
    lemmas = [token.lemma_ for token in spacy_doc if not token.is_punct and not token.is_space]
    lemmatized_docs.append(" ".join(lemmas))
docs = lemmatized_docs

## **2. FIT AND TRAIN MODEL**
#### a. Pre-calculate embeddings to speed up tuning later
##### *This cell takes the longest time to run*

In [9]:
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device="cuda")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
batch_size = 32
embeddings = embedding_model.encode(
    docs,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/242 [00:00<?, ?it/s]

In [19]:
representation_model = KeyBERTInspired()
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

topic_model = BERTopic(representation_model=representation_model,
                       embedding_model=embedding_model,
                       hdbscan_model=hdbscan_model,
                       ctfidf_model=ctfidf_model,
                       language="english",
                       calculate_probabilities=False, verbose=True)

In [20]:
topics, probs = topic_model.fit_transform(docs, embeddings)
freq = topic_model.get_topic_info()

2025-04-09 02:51:43,035 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-04-09 02:51:56,139 - BERTopic - Dimensionality - Completed ✓
2025-04-09 02:51:56,140 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-04-09 02:51:56,606 - BERTopic - Cluster - Completed ✓
2025-04-09 02:51:56,617 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-04-09 02:52:17,711 - BERTopic - Representation - Completed ✓


In [24]:
freq.to_csv("firstFinetunedTopics.csv", index=False)